# Analise Macroeconomica — Brasil

Analise do cenario macroeconomico brasileiro com dados reais do BCB, IBGE e analyzers.

Fontes:
- **BCBFetcher**: Selic, CDI, IPCA, IGP-M, PTAX, TR, INPC, poupanca
- **IBGEFetcher**: PIB, desemprego, producao industrial
- **FiscalAnalyzer**: divida/PIB, resultado primario, trajetoria fiscal
- **CurrencyAnalyzer**: USD/BRL, carry trade, cambio real efetivo

**Nota**: Este notebook faz chamadas reais a APIs publicas (BCB, IBGE). Nao requer API keys.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.titlesize"] = 14

## 1. Politica Monetaria — Selic e CDI

A taxa Selic e a principal ferramenta de politica monetaria do BCB.
O CDI acompanha de perto a Selic e serve como benchmark para renda fixa.

In [ ]:
from carteira_auto.data.fetchers import BCBFetcher

bcb = BCBFetcher()

# Selic meta — serie historica de 5 anos
selic = bcb.get_selic(period_days=5 * 365)
print(f"Selic atual: {selic['valor'].iloc[-1]:.2f}% a.a.")
print(f"Periodo: {selic['data'].iloc[0].date()} a {selic['data'].iloc[-1].date()}")
print(f"Registros: {len(selic)}")

In [ ]:
# CDI
cdi = bcb.get_cdi(period_days=5 * 365)
print(f"CDI atual: {cdi['valor'].iloc[-1]:.4f}% a.d.")

# Visualizar Selic
fig, ax = plt.subplots()
ax.plot(selic["data"], selic["valor"], linewidth=2, color="#1f77b4")
ax.set_title("Taxa Selic Meta — Ultimos 5 Anos")
ax.set_ylabel("% a.a.")
ax.axhline(y=selic["valor"].iloc[-1], color="red", linestyle="--", alpha=0.5,
           label=f"Atual: {selic['valor'].iloc[-1]:.2f}%")
ax.legend()
plt.tight_layout()
plt.show()

## 2. Inflacao — IPCA e IGP-M

In [ ]:
# IPCA mensal
ipca = bcb.get_ipca(period_days=3 * 365)
igpm = bcb.get_igpm(period_days=3 * 365)

print(f"IPCA ultimo mes: {ipca['valor'].iloc[-1]:.2f}%")
print(f"IGP-M ultimo mes: {igpm['valor'].iloc[-1]:.2f}%")

# IPCA acumulado 12 meses
ipca_12m = ipca.tail(12)
ipca_acum = ((1 + ipca_12m["valor"] / 100).prod() - 1) * 100
print(f"\nIPCA acumulado 12m: {ipca_acum:.2f}%")

In [ ]:
# Visualizar IPCA mensal — barras por mês
fig, ax = plt.subplots()
colors = ["#d62728" if v > 0 else "#2ca02c" for v in ipca["valor"]]
ax.bar(range(len(ipca)), ipca["valor"], color=colors, alpha=0.8, width=0.8)

# Eixo x: rótulos mensais (MMM/YY)
labels = ipca["data"].dt.strftime("%b/%y")
ax.set_xticks(range(0, len(ipca), 3))
ax.set_xticklabels(labels.iloc[::3], rotation=45, ha="right")

ax.set_title("IPCA Mensal — Últimos 3 Anos")
ax.set_ylabel("% mensal")
ax.axhline(y=0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# IPCA acumulado 12 meses (rolling)
ipca_rolling = ipca.copy()
ipca_rolling["acum_12m"] = (
    (1 + ipca_rolling["valor"] / 100).rolling(12).apply(lambda x: x.prod() - 1) * 100
)
ipca_rolling = ipca_rolling.dropna(subset=["acum_12m"])

fig, ax = plt.subplots()
ax.plot(ipca_rolling["data"], ipca_rolling["acum_12m"], linewidth=2, color="#d62728")
ax.axhline(y=3.0, color="green", linestyle="--", alpha=0.6, label="Centro da meta (3.0%)")
ax.axhspan(1.5, 4.5, alpha=0.08, color="green", label="Banda da meta")
ax.set_title("IPCA Acumulado 12 Meses vs Meta de Inflação")
ax.set_ylabel("% acum. 12m")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# IPCA vs IGP-M — comparativo de inflação
fig, ax = plt.subplots()
ax.plot(ipca["data"], ipca["valor"], label="IPCA", linewidth=2, color="#d62728")
ax.plot(igpm["data"], igpm["valor"], label="IGP-M", linewidth=2, color="#ff7f0e")
ax.axhline(y=0, color="black", linewidth=0.5)
ax.set_title("IPCA vs IGP-M — Variação Mensal")
ax.set_ylabel("% mensal")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Cambio — USD/BRL e PTAX

In [ ]:
# PTAX compra (serie de 2 anos)
ptax = bcb.get_ptax(period_days=2 * 365)
print(f"PTAX atual: R$ {ptax['valor'].iloc[-1]:.4f}")

# Variacao 12m
if len(ptax) >= 252:
    var_12m = (ptax["valor"].iloc[-1] / ptax["valor"].iloc[-252] - 1) * 100
    print(f"Variação 12m: {var_12m:+.1f}%")

# Visualizar com cores dinâmicas e eixo y ajustado
fig, ax = plt.subplots()

# Cor baseada na tendência: verde se valorizou (caiu), vermelho se desvalorizou (subiu)
primeiro = ptax["valor"].iloc[0]
ultimo = ptax["valor"].iloc[-1]
cor_principal = "#2ca02c" if ultimo <= primeiro else "#d62728"
label_tend = "Valorização BRL" if ultimo <= primeiro else "Desvalorização BRL"

ax.plot(ptax["data"], ptax["valor"], linewidth=1.5, color=cor_principal, label=label_tend)
ax.fill_between(ptax["data"], ptax["valor"], alpha=0.1, color=cor_principal)

# Eixo y ajustado ao range real dos dados (não começa em 0)
y_min = ptax["valor"].min() * 0.98
y_max = ptax["valor"].max() * 1.02
ax.set_ylim(y_min, y_max)

ax.axhline(y=ultimo, color="gray", linestyle="--", alpha=0.5,
           label=f"Atual: R$ {ultimo:.2f}")
ax.set_title("PTAX Compra (USD/BRL) — Últimos 2 Anos")
ax.set_ylabel("R$/USD")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Atividade Economica — PIB e Desemprego

In [ ]:
from carteira_auto.data.fetchers import IBGEFetcher

ibge = IBGEFetcher()

# PIB trimestral
pib = ibge.get_pib(quarters=12)
print(f"PIB: {len(pib)} trimestres")
print(pib.tail(4))

In [ ]:
# Desemprego (PNAD Continua)
desemprego = ibge.get_unemployment(quarters=12)
print(f"Desemprego: {len(desemprego)} trimestres")
print(desemprego.tail(4))

In [ ]:
# PIB contínuo — série longa com linha temporal
pib_long = ibge.get_pib(quarters=20)

# Converter código de período (ex: "202501") para data real
_q_to_m = {"01": "01", "02": "04", "03": "07", "04": "10"}
pib_long["mes"] = pib_long["periodo_codigo"].astype(str).str[4:].map(_q_to_m)
pib_long["ano"] = pib_long["periodo_codigo"].astype(str).str[:4]
pib_long["date"] = pd.to_datetime(
    pib_long["ano"] + "-" + pib_long["mes"] + "-01", errors="coerce"
)
pib_long = pib_long.dropna(subset=["date"]).sort_values("date")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Série contínua (linha + área)
ax1.plot(pib_long["date"], pib_long["valor"], "o-", linewidth=2,
         markersize=5, color="#1f77b4")
ax1.fill_between(pib_long["date"], pib_long["valor"], 0,
                 where=[v >= 0 for v in pib_long["valor"]],
                 alpha=0.15, color="#2ca02c", label="Crescimento")
ax1.fill_between(pib_long["date"], pib_long["valor"], 0,
                 where=[v < 0 for v in pib_long["valor"]],
                 alpha=0.15, color="#d62728", label="Contração")
ax1.axhline(y=0, color="black", linewidth=0.8)
ax1.set_title("PIB — Variação Trimestral YoY (5 Anos)")
ax1.set_ylabel("% vs mesmo trimestre ano anterior")
ax1.legend()

# Desemprego (série contínua)
_q_to_m2 = {"10": "01", "11": "04", "12": "07", "01": "10"}
desemprego_long = ibge.get_unemployment(quarters=20)
if not desemprego_long.empty:
    try:
        desemprego_long["mes"] = desemprego_long["periodo_codigo"].astype(str).str[4:].map(
            lambda x: {"10": "01", "11": "04", "12": "07", "01": "10"}.get(x, "01")
        )
        desemprego_long["ano"] = desemprego_long["periodo_codigo"].astype(str).str[:4]
        desemprego_long["date"] = pd.to_datetime(
            desemprego_long["ano"] + "-" + desemprego_long["mes"] + "-01", errors="coerce"
        )
        desemprego_long = desemprego_long.dropna(subset=["date"]).sort_values("date")
        ax2.plot(desemprego_long["date"], desemprego_long["valor"], "o-",
                 linewidth=2, markersize=5, color="#d62728")
        ax2.fill_between(desemprego_long["date"], desemprego_long["valor"],
                         desemprego_long["valor"].min() * 0.95, alpha=0.1, color="#d62728")
        y_min = desemprego_long["valor"].min() * 0.95
        y_max = desemprego_long["valor"].max() * 1.05
        ax2.set_ylim(y_min, y_max)
    except Exception:
        pass

ax2.set_title("Desemprego — Taxa de Desocupação PNAD (5 Anos)")
ax2.set_ylabel("% da PEA")

plt.tight_layout()
plt.show()

## 5. Juros Reais — Selic vs IPCA

In [ ]:
# Juro real = Selic - IPCA acum. 12m (aproximacao)
selic_atual = selic["valor"].iloc[-1]
juro_real = selic_atual - ipca_acum

print(f"Selic: {selic_atual:.2f}% a.a.")
print(f"IPCA 12m: {ipca_acum:.2f}%")
print(f"Juro real aproximado: {juro_real:.2f}% a.a.")
print(f"\nInterpretacao: {'Restritivo' if juro_real > 3 else 'Neutro' if juro_real > 0 else 'Expansionista'}")

In [ ]:
# Visualizar comparativo: Selic nominal vs IPCA 12m vs Juro Real
fig, ax = plt.subplots(figsize=(8, 5))

categorias = ["Selic nominal", "IPCA 12m", "Juro real"]
valores = [selic_atual, ipca_acum, juro_real]
cores = ["#1f77b4", "#d62728", "#2ca02c"]

bars = ax.bar(categorias, valores, color=cores, alpha=0.85, width=0.5)
for bar, val in zip(bars, valores, strict=False):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
            f"{val:.1f}%", ha="center", fontweight="bold", fontsize=12)

ax.set_title("Selic Nominal vs IPCA vs Juro Real")
ax.set_ylabel("% a.a.")
ax.axhline(y=0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

## 6. Analise Fiscal — FiscalAnalyzer

O FiscalAnalyzer busca 5 series do BCB e classifica a trajetoria fiscal
com gradacao: stable → warning → critical → severe.

In [ ]:
from carteira_auto.analyzers import FiscalAnalyzer
from carteira_auto.core.engine import PipelineContext

fiscal = FiscalAnalyzer()
ctx = PipelineContext()
ctx = fiscal.run(ctx)

fm = ctx["fiscal_metrics"]
print("=== Métricas Fiscais ===")
print(f"Dívida bruta/PIB:    {fm.divida_bruta_pib}%")
print(f"Dívida líquida/PIB:  {fm.divida_liquida_pib}%")
print(f"Resultado primário:  {fm.resultado_primario_pib}%")
print(f"Juros nominais/PIB:  {fm.juros_nominais_pib}%")
print(f"Var. dívida 12m:     {fm.divida_bruta_pib_change_12m} pp")
print(f"Trajetória:          {fm.fiscal_trajectory}")
print(f"\nSumário: {fm.summary}")

In [ ]:
# Painel fiscal visual
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Composição fiscal (% do PIB)
labels_fiscal = ["Dív. bruta", "Dív. líquida", "Juros nom.", "Res. primário"]
valores_fiscal = [
    fm.divida_bruta_pib or 0,
    fm.divida_liquida_pib or 0,
    fm.juros_nominais_pib or 0,
    fm.resultado_primario_pib or 0,
]
cores_fiscal = ["#d62728", "#ff7f0e", "#9467bd",
                "#2ca02c" if (fm.resultado_primario_pib or 0) > 0 else "#d62728"]

bars = ax1.barh(labels_fiscal, valores_fiscal, color=cores_fiscal, alpha=0.85)
for bar, val in zip(bars, valores_fiscal, strict=False):
    ax1.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
             f"{val:.1f}%", va="center", fontweight="bold")
ax1.set_title("Indicadores Fiscais (% do PIB)")
ax1.set_xlabel("% do PIB")

# Faixas de risco da dívida bruta/PIB
divida = fm.divida_bruta_pib or 0
faixas = [75, 80, 85, 100]
faixa_labels = ["Estável\n(<75%)", "Atenção\n(75-80%)", "Crítico\n(80-85%)", "Severo\n(>85%)"]
faixa_cores = ["#2ca02c", "#ff7f0e", "#d62728", "#8b0000"]

ax2.barh(faixa_labels, faixas, color=faixa_cores, alpha=0.3, height=0.6)
ax2.axvline(x=divida, color="black", linewidth=3,
            label=f"Atual: {divida:.1f}%")
ax2.set_title(f"Dívida Bruta/PIB — Trajetória: {fm.fiscal_trajectory}")
ax2.set_xlabel("% do PIB")
ax2.legend(fontsize=12)

plt.tight_layout()
plt.show()

## 7. Analise Cambial — CurrencyAnalyzer

O CurrencyAnalyzer busca PTAX, DXY, Selic e Fed Funds
para calcular carry spread e tendencia cambial.

**Nota**: requer `FRED_API_KEY` para Fed Funds Rate. Sem a key, carry_spread sera None.

In [ ]:
from carteira_auto.analyzers import CurrencyAnalyzer

currency = CurrencyAnalyzer()
ctx = PipelineContext()
ctx = currency.run(ctx)

cm = ctx["currency_metrics"]
print("=== Metricas Cambiais ===")
print(f"USD/BRL (PTAX): R$ {cm.usd_brl}")
print(f"Variacao 1m: {cm.usd_brl_change_1m}%")
print(f"Variacao 3m: {cm.usd_brl_change_3m}%")
print(f"Variacao 12m: {cm.usd_brl_change_12m}%")
print(f"DXY: {cm.dxy}")
print(f"Selic: {cm.selic_rate}% | Fed Funds: {cm.fed_funds_rate}%")
print(f"Carry spread: {cm.carry_spread} pp")
print(f"Cambio real efetivo: {cm.taxa_cambio_real_efetiva}")
print(f"\nSumario: {cm.summary}")

In [ ]:
# Gráficos CurrencyAnalyzer
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Variação USD/BRL por período
ax1 = axes[0]
periodos = ["1 mês", "3 meses", "12 meses"]
variacoes = [cm.usd_brl_change_1m, cm.usd_brl_change_3m, cm.usd_brl_change_12m]
variacoes = [v if v is not None else 0 for v in variacoes]
cores_var = ["#d62728" if v > 0 else "#2ca02c" for v in variacoes]
bars = ax1.bar(periodos, variacoes, color=cores_var, alpha=0.85, width=0.5)
for bar, val in zip(bars, variacoes, strict=False):
    ax1.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + (0.1 if val >= 0 else -0.3),
             f"{val:+.1f}%", ha="center", fontweight="bold")
ax1.axhline(y=0, color="black", linewidth=0.5)
ax1.set_title("USD/BRL — Variação por Período\n(+ = desvalorização BRL)")
ax1.set_ylabel("%")

# 2. DXY vs PTAX (posição atual relativa)
ax2 = axes[1]
indicadores = ["USD/BRL\natual", "USD/BRL\n(DXY 1m)"]
valores_cam = [cm.usd_brl or 0, cm.dxy_change_1m or 0]
ax2.bar(["USD/BRL\natual"], [cm.usd_brl or 0], color="#d62728", alpha=0.85, width=0.4)
ax2_right = ax2.twinx()
ax2_right.bar(["DXY\nvar. 1m"], [cm.dxy_change_1m or 0],
              color="#9467bd", alpha=0.85, width=0.4)
ax2.set_ylabel("R$/USD", color="#d62728")
ax2_right.set_ylabel("DXY var. %", color="#9467bd")
ax2.set_title(f"USD/BRL: R${cm.usd_brl:.2f} | DXY: {cm.dxy or 'N/A'}")

# 3. Carry trade spread
ax3 = axes[2]
if cm.selic_rate and cm.fed_funds_rate:
    taxas = [cm.selic_rate, cm.fed_funds_rate, cm.carry_spread or 0]
    labels_carry = ["Selic\n(BR)", "Fed Funds\n(EUA)", "Carry\nSpread"]
    cores_carry = ["#1f77b4", "#d62728", "#2ca02c" if (cm.carry_spread or 0) > 5 else "#ff7f0e"]
    bars = ax3.bar(labels_carry, taxas, color=cores_carry, alpha=0.85, width=0.5)
    for bar, val in zip(bars, taxas, strict=False):
        ax3.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
                 f"{val:.1f}%", ha="center", fontweight="bold")
    ax3.set_title("Carry Trade — Selic vs Fed Funds")
    ax3.set_ylabel("% a.a.")
else:
    ax3.text(0.5, 0.5, "Fed Funds N/A\n(requer FRED_API_KEY)",
             ha="center", va="center", transform=ax3.transAxes)
    ax3.set_title("Carry Trade")

plt.suptitle("CurrencyAnalyzer — Análise Cambial", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 8. MacroAnalyzer — Contexto Consolidado

O MacroAnalyzer consolida 11 indicadores macro em um unico `MacroContext`.

In [ ]:
from carteira_auto.analyzers import MacroAnalyzer

macro = MacroAnalyzer()
ctx = PipelineContext()
ctx = macro.run(ctx)

mc = ctx["macro_context"]
print("=== Contexto Macro Consolidado ===")
print(f"Selic: {mc.selic}% | CDI: {mc.cdi:.2f}%")
print(f"IPCA: {mc.ipca:.2f}% | IGP-M: {mc.igpm:.2f}%")
print(f"PTAX: R$ {mc.cambio}")
print(f"Poupança: {mc.poupanca:.2f}% | TR: {mc.tr:.2f}%")
print(f"PIB crescimento: {mc.pib_growth}%")
print(f"Desocupação: {mc.desocupacao}%")
print(f"\nSumário: {mc.summary}")

# Erros parciais (se houve falhas)
if ctx.has_errors:
    print(f"\nErros parciais: {ctx.errors}")

In [ ]:
# Gráficos CommodityAnalyzer
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Preços atuais das principais commodities
labels_comm = ["Brent\n(US$/barril)", "Ouro\n(US$/oz)", "Soja\n(US$/bushel)",
               "Milho\n(US$/bushel)", "Trigo\n(US$/bushel)"]
precos_comm = [ccm.oil_brent or 0, (ccm.gold or 0) / 10,  # escala ouro / 10 para comparar
               ccm.soybean or 0, ccm.corn or 0, ccm.wheat or 0]
cores_comm = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]

bars = ax1.bar(labels_comm, precos_comm, color=cores_comm, alpha=0.85)
ax1.set_title("Preços de Commodities (Ouro ÷10 para escala)")
ax1.set_ylabel("US$")
for bar, val in zip(bars, precos_comm, strict=False):
    if val > 0:
        ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                 f"{val:.0f}", ha="center", fontsize=9)

# Variações 1 mês (barômetro de curto prazo)
labels_var = ["Brent 1m", "Brent 12m", "Ouro 1m", "Ouro 12m",
              "Soja 1m", "Índice\n3m"]
vars_comm = [
    ccm.oil_change_1m or 0, ccm.oil_change_12m or 0,
    ccm.gold_change_1m or 0, ccm.gold_change_12m or 0,
    ccm.soybean_change_1m or 0, ccm.commodity_index_change_3m or 0,
]
cores_v = ["#2ca02c" if v >= 0 else "#d62728" for v in vars_comm]
bars2 = ax2.bar(labels_var, vars_comm, color=cores_v, alpha=0.85)
ax2.axhline(y=0, color="black", linewidth=0.8)
for bar, val in zip(bars2, vars_comm, strict=False):
    offset = 0.1 if val >= 0 else -0.4
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + offset,
             f"{val:+.1f}%", ha="center", fontsize=9, fontweight="bold")
ciclo_cor = {"expansão": "🟢", "contração": "🔴", "contração_avançada": "🔴",
             "trough": "🟡"}.get(ccm.cycle_signal or "", "⚪")
ax2.set_title(f"Variações — Ciclo: {ciclo_cor} {ccm.cycle_signal or 'N/A'}")
ax2.set_ylabel("% variação")

plt.tight_layout()
plt.show()

In [ ]:
from carteira_auto.analyzers import CommodityAnalyzer

commodity = CommodityAnalyzer()
ctx_comm = PipelineContext()
ctx_comm = commodity.run(ctx_comm)

ccm = ctx_comm["commodity_metrics"]
print("=== Commodities ===")
print(f"Petróleo Brent: US${ccm.oil_brent} | WTI: US${ccm.oil_wti}")
print(f"  Variação 1m: {ccm.oil_change_1m}% | 12m: {ccm.oil_change_12m}%")
print(f"Ouro: US${ccm.gold} | Variação 1m: {ccm.gold_change_1m}% | 12m: {ccm.gold_change_12m}%")
print(f"Soja: US${ccm.soybean} | Milho: US${ccm.corn} | Trigo: US${ccm.wheat}")
print(f"Índice composto (var. 3m): {ccm.commodity_index_change_3m}%")
print(f"Sinal de ciclo: {ccm.cycle_signal}")
print(f"\nSumário: {ccm.summary}")

## 9. CommodityAnalyzer — Mercado de Commodities

Commodities relevantes para a economia brasileira: petróleo, ouro, soja.
O Brasil é exportador líquido — alta de commodities tende a fortalecer o BRL.

## Resumo do Cenario Macro

Os dados acima permitem avaliar:

| Dimensao | Indicadores | Analise |
|----------|-------------|---------|
| **Politica monetaria** | Selic, CDI, juro real | Restritivo se juro real > 3% |
| **Inflacao** | IPCA, IGP-M | Dentro da meta? Tendencia? |
| **Cambio** | PTAX, DXY, carry | Apreciacao ou depreciacao do BRL? |
| **Atividade** | PIB, desemprego | Ciclo de expansao ou contracao? |
| **Fiscal** | Divida/PIB, primario | Trajetoria sustentavel? |